In [3]:
import numpy as np, pandas as pd 
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_validate, GridSearchCV 
from sklearn.compose import ColumnTransformer 
from sklearn.pipeline import Pipeline 
from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import OneHotEncoder, StandardScaler 
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge 
from sklearn.neighbors import KNeighborsClassifier 
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor 
from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_error 
clf_df = pd.read_csv("data/classification_escalation.csv") 
reg_df = pd.read_csv("data/resolution_time_regression.csv") 
display(clf_df.head(), reg_df.head())

,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
0,T0000,2.0,15.0,Medium,Email,61.818896,0
1,T0001,9.0,6.0,Critical,Chat,110.024317,0
2,T0002,10.0,8.0,High,Phone,36.311244,0
3,T0003,2.0,1.0,Medium,Phone,85.127855,0
4,T0004,8.0,14.0,Medium,Chat,18.225059,0


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
0,R0000,7.0,10.0,Medium,Chat,32.638377,8.38724
1,R0001,4.0,15.0,Medium,Chat,45.273760,0.50000
2,R0002,4.0,13.0,Low,Portal,44.878323,0.50000
3,R0003,1.0,10.0,High,Phone,40.095034,0.50000
4,R0004,3.0,15.0,Low,Portal,47.451134,0.50000


In [4]:
print("CLASSIFICATION DATASET")
print(clf_df.shape)
print(clf_df.info())
print(clf_df.isnull().sum())

print("\nREGRESSION DATASET")
print(reg_df.shape)
print(reg_df.info())
print(reg_df.isnull().sum())

CLASSIFICATION DATASET
(900, 7)
<class 'pandas.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ticket_id               900 non-null    str    
 1   complexity_score        878 non-null    float64
 2   team_experience_years   864 non-null    float64
 3   priority                887 non-null    str    
 4   channel                 891 non-null    str    
 5   customer_tenure_months  869 non-null    float64
 6   escalated               900 non-null    int64  
dtypes: float64(3), int64(1), str(3)
memory usage: 49.3 KB
None
ticket_id                  0
complexity_score          22
team_experience_years     36
priority                  13
channel                    9
customer_tenure_months    31
escalated                  0
dtype: int64

REGRESSION DATASET
(800, 7)
<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 7 column

In [5]:
print("Classification target distribution:")
print(clf_df["escalated"].value_counts())
print(clf_df["escalated"].value_counts(normalize=True))

print("\nRegression target statistics:")
print(reg_df["resolution_time_hours"].describe())

Classification target distribution:
escalated
0    709
1    191
Name: count, dtype: int64
escalated
0    0.787778
1    0.212222
Name: proportion, dtype: float64

Regression target statistics:
count    800.000000
mean       5.873160
std        4.144974
min        0.500000
25%        2.631319
50%        5.387386
75%        8.345903
max       27.488229
Name: resolution_time_hours, dtype: float64


In [6]:
X_clf = clf_df.drop(columns=["ticket_id", "escalated"])
y_clf = clf_df["escalated"]

print("Features shape:", X_clf.shape)
print("Target shape:", y_clf.shape)

display(X_clf.head())

Features shape: (900, 5)
Target shape: (900,)


,complexity_score,team_experience_years,priority,channel,customer_tenure_months
0,2.0,15.0,Medium,Email,61.818896
1,9.0,6.0,Critical,Chat,110.024317
2,10.0,8.0,High,Phone,36.311244
3,2.0,1.0,Medium,Phone,85.127855
4,8.0,14.0,Medium,Chat,18.225059


In [7]:
numeric_features = X_clf.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_clf.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

Numerical features: ['complexity_score', 'team_experience_years', 'customer_tenure_months']
Categorical features: ['priority', 'channel']


In [8]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf,
    y_clf,
    test_size=0.2,
    random_state=42,
    stratify=y_clf
)

print("Training set:", X_train_clf.shape)
print("Testing set:", X_test_clf.shape)

print("\nTraining distribution:")
print(y_train_clf.value_counts(normalize=True))

print("\nTesting distribution:")
print(y_test_clf.value_counts(normalize=True))

Training set: (720, 5)
Testing set: (180, 5)

Training distribution:
escalated
0    0.7875
1    0.2125
Name: proportion, dtype: float64

Testing distribution:
escalated
0    0.788889
1    0.211111
Name: proportion, dtype: float64


### Why Split Before Preprocessing?

Preprocessing must be performed after splitting the dataset to prevent data leakage. If we calculate imputation statistics or scaling parameters using the entire dataset, information from the test set influences the preprocessing process.

To prevent this, we will place all preprocessing operations inside an sklearn Pipeline. During cross-validation, each preprocessing transformation is fitted exclusively on the training fold and then applied to its corresponding validation fold.

This ensures that validation and test observations remain unseen during model training and preprocessing.

### Preprocessing Strategy

Our classification dataset contains numerical and categorical features with missing values.

**Numerical features:**
- SimpleImputer(strategy="median"): Replaces missing numerical values with the median of the training data.
- StandardScaler(): Standardizes numerical features to have approximately zero mean and unit variance.

Scaling is particularly important for k-NN because it calculates distances between observations. Without scaling, features with larger numerical ranges could disproportionately influence the distance calculations.

**Categorical features:**
- SimpleImputer(strategy="most_frequent"): Replaces missing categorical values with the most frequent category.
- OneHotEncoder(handle_unknown="ignore"): Converts categorical values into binary features. The unknown-category option prevents errors when previously unseen categories appear during prediction.

In [9]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [10]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "KNN": KNeighborsClassifier(),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    )
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [11]:
results = {}

for name, model in models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", model)
        ]
    )

    scores = cross_validate(
        pipeline,
        X_train_clf,
        y_train_clf,
        cv=cv,
        scoring="f1",
        return_train_score=True
    )

    results[name] = {
        "Mean F1": scores["test_score"].mean(),
        "Std F1": scores["test_score"].std()
    }

results_df = pd.DataFrame(results).T

display(results_df.sort_values("Mean F1", ascending=False))

,Mean F1,Std F1
Logistic Regression,0.392651,0.062747
KNN,0.360835,0.050295
Decision Tree,0.330666,0.058126


### Cross-Validation Results and Initial Model Selection

Logistic Regression achieved the highest mean F1-score (0.3927), outperforming KNN (0.3608) and Decision Tree (0.3307).

KNN exhibited the lowest standard deviation (0.0503), indicating relatively consistent performance across folds. However, its average predictive performance was lower than that of Logistic Regression.

Logistic Regression was therefore selected as the initial candidate for hyperparameter tuning because it achieved the highest average F1-score.

The standard deviation of 0.0627 indicates some variation in its performance across validation folds, suggesting that its predictions are somewhat sensitive to the particular training and validation partitions.

### Logistic Regression Hyperparameter Tuning

We will use GridSearchCV to optimize Logistic Regression using the following hyperparameters:

- **C:** Controls the strength of regularization. Smaller values apply stronger regularization, while larger values allow the model to fit the training data more freely.
- **class_weight:** Determines the relative importance assigned to each class. We will compare the default weighting with balanced weighting to investigate whether compensating for class imbalance improves F1-score.

The search space contains five C values (0.01, 0.1, 1, 10, 100) and two class-weight configurations (None and balanced), resulting in ten candidate configurations.

In [12]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)

param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100],
    "classifier__class_weight": [None, "balanced"]
}

grid_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)

In [13]:
grid_search.fit(X_train_clf, y_train_clf)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__C': [0.01, 0.1, ...], 'classifier__class_weight': [None, 'balanced']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all 

In [14]:
print("Best parameters:")
print(grid_search.best_params_)

print("\nBest cross-validation F1:")
print(grid_search.best_score_)

Best parameters:
{'classifier__C': 0.1, 'classifier__class_weight': 'balanced'}

Best cross-validation F1:
0.5199932976178999


In [15]:
tuning_results = pd.DataFrame(grid_search.cv_results_)

tuning_results = tuning_results[
    [
        "param_classifier__C",
        "param_classifier__class_weight",
        "mean_test_score",
        "std_test_score",
        "mean_train_score"
    ]
].sort_values("mean_test_score", ascending=False)

display(tuning_results)

,param_classifier__C,param_classifier__class_weight,mean_test_score,std_test_score,mean_train_score
3,0.10,balanced,0.519993,0.042344,0.534184
5,1.00,balanced,0.515450,0.034507,0.529524
1,0.01,balanced,0.515422,0.042798,0.520235
9,100.00,balanced,0.508308,0.039741,0.529521
7,10.00,balanced,0.508308,0.039741,0.529817
6,10.00,NaN,0.422593,0.088225,0.444438
8,100.00,NaN,0.422593,0.088225,0.444438
4,1.00,NaN,0.392651,0.062747,0.439963
2,0.10,NaN,0.327873,0.046086,0.354718
0,0.01,NaN,0.000000,0.000000,0.000000


### Hyperparameter Tuning Results

GridSearchCV selected Logistic Regression with C=0.1 and class_weight="balanced", achieving a mean cross-validation F1-score of 0.5200 and a standard deviation of 0.0423.

This represents an improvement over the initial Logistic Regression configuration, which achieved a mean F1-score of 0.3927.

The improvement suggests that stronger regularization and balanced class weighting helped the model identify escalated tickets more effectively.

The relatively small difference between mean training F1 (0.5342) and mean validation F1 (0.5200) suggests limited overfitting across the validation folds.

The results also indicate that class weighting was particularly beneficial for this imbalanced classification problem.

In [16]:
best_clf = grid_search.best_estimator_

y_pred_clf = best_clf.predict(X_test_clf)

print("Test Classification Report:")
print(classification_report(y_test_clf, y_pred_clf))

print("Confusion Matrix:")
print(confusion_matrix(y_test_clf, y_pred_clf))

Test Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.82      0.87       142
           1       0.52      0.71      0.60        38

    accuracy                           0.80       180
   macro avg       0.72      0.77      0.73       180
weighted avg       0.83      0.80      0.81       180

Confusion Matrix:
[[117  25]
 [ 11  27]]


In [17]:
from sklearn.metrics import f1_score

test_f1 = f1_score(y_test_clf, y_pred_clf)

print("Test F1-score:", test_f1)

Test F1-score: 0.6


### Final Classification Model Evaluation

After hyperparameter tuning, the selected Logistic Regression pipeline was evaluated on the held-out test set.

The model achieved:
- Accuracy: 0.80
- Precision for escalated tickets: 0.52
- Recall for escalated tickets: 0.71
- F1-score for escalated tickets: 0.60

The confusion matrix was:

| | Predicted 0 | Predicted 1 |
|---|---:|---:|
| Actual 0 | 117 | 25 |
| Actual 1 | 11 | 27 |

The model correctly identified 27 out of 38 escalated tickets, resulting in a recall of approximately 71%.

However, it also produced 25 false positives, meaning that some non-escalated tickets would receive unnecessary escalation alerts. Additionally, 11 actual escalations were missed.

This demonstrates a trade-off between identifying more escalated tickets and generating additional false alarms.

The final test F1-score of 0.60 is higher than the cross-validation mean of approximately 0.52. This difference may result from natural variation between the test sample and the validation folds.

Overall, the tuned Logistic Regression model is preferable to the initial candidates based on its cross-validation performance. However, its precision indicates that further improvement may be necessary before deployment.

## Experiment B - Regression

In [18]:
X_reg = reg_df.drop(
    columns=["ticket_id", "resolution_time_hours"]
)

y_reg = reg_df["resolution_time_hours"]

print("Features shape:", X_reg.shape)
print("Target shape:", y_reg.shape)

display(X_reg.head())

Features shape: (800, 5)
Target shape: (800,)


,complexity_score,team_experience_years,priority,channel,customer_tenure_months
0,7.0,10.0,Medium,Chat,32.638377
1,4.0,15.0,Medium,Chat,45.273760
2,4.0,13.0,Low,Portal,44.878323
3,1.0,10.0,High,Phone,40.095034
4,3.0,15.0,Low,Portal,47.451134


In [19]:
numeric_features_reg = X_reg.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features_reg = X_reg.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

print("Numerical features:", numeric_features_reg)
print("Categorical features:", categorical_features_reg)

Numerical features: ['complexity_score', 'team_experience_years', 'customer_tenure_months']
Categorical features: ['priority', 'channel']


In [20]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

print("Training set:", X_train_reg.shape)
print("Testing set:", X_test_reg.shape)

Training set: (640, 5)
Testing set: (160, 5)


### Evaluation Metric: Mean Absolute Error (MAE)

MAE measures the average absolute difference between actual and predicted resolution times.

It is appropriate for this problem because it expresses prediction errors in hours, making the results easy to interpret in the context of ticket resolution.

For example, an MAE of 2 hours means that predictions differ from actual resolution times by an average of 2 hours.

Unlike RMSE, MAE does not square prediction errors, making it less sensitive to extreme prediction errors and potential outliers.

Lower MAE values indicate better predictive performance.

### Regression Preprocessing Strategy

We will use a ColumnTransformer to preprocess numerical and categorical features separately.

**Numerical features:**
- Median imputation to handle missing numerical observations.
- StandardScaler to standardize numerical features.

**Categorical features:**
- Most-frequent imputation to replace missing categorical values.
- One-hot encoding to convert categorical features into numerical representations.

The preprocessing operations will be included inside each model pipeline to prevent data leakage during cross-validation.

Standardization is particularly relevant for Ridge Regression because its regularization penalizes coefficient magnitudes. Scaling ensures that the regularization penalty is not disproportionately influenced by the original units of different numerical features.

In [21]:
numeric_transformer_reg = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer_reg = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor_reg = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_reg, numeric_features_reg),
        ("cat", categorical_transformer_reg, categorical_features_reg)
    ]
)

In [22]:
models_reg = {
    "Linear Regression": LinearRegression(),

    "Ridge Regression": Ridge(),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    )
}

cv_reg = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [23]:
results_reg = {}

for name, model in models_reg.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor_reg),
            ("regressor", model)
        ]
    )

    scores = cross_validate(
        pipeline,
        X_train_reg,
        y_train_reg,
        cv=cv_reg,
        scoring="neg_mean_absolute_error",
        return_train_score=True
    )

    results_reg[name] = {
        "Mean MAE": -scores["test_score"].mean(),
        "Std MAE": scores["test_score"].std(),
        "Mean Train MAE": -scores["train_score"].mean()
    }

results_reg_df = pd.DataFrame(results_reg).T

display(results_reg_df.sort_values("Mean MAE"))

,Mean MAE,Std MAE,Mean Train MAE
Ridge Regression,2.035127,0.171404,1.985489
Linear Regression,2.035134,0.171650,1.985063
Decision Tree,3.239097,0.053807,-0.000000


### Cross-Validation Results and Initial Model Selection

Ridge Regression achieved the lowest mean validation MAE of 2.0351 hours, closely followed by Linear Regression at 2.0351 hours.

The difference between these two models is negligible, indicating that regularization has little effect on predictive performance with the current dataset and default Ridge configuration.

Decision Tree Regression performed considerably worse, achieving a mean validation MAE of 3.2391 hours. Although its training MAE was approximately zero, its validation error was substantially higher. This indicates severe overfitting: the tree memorizes training observations but does not generalize effectively to unseen data.

Ridge Regression was selected for hyperparameter tuning because it achieved the lowest mean validation MAE and provides regularization as a mechanism for controlling model complexity.

The results suggest that a linear relationship may provide a more effective representation of the available features than the unrestricted decision tree.

### Ridge Regression Hyperparameter Tuning

We will use GridSearchCV to optimize the regularization strength (`alpha`) of Ridge Regression.

The search space consists of six values: 0.001, 0.01, 0.1, 1, 10, and 100.

This range allows us to investigate how different regularization strengths affect predictive performance.

We will use the same five-fold KFold configuration and negative MAE scoring used in the initial model comparison.

The configuration with the lowest mean validation MAE will be selected.

In [24]:
ridge_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_reg),
        ("regressor", Ridge())
    ]
)

ridge_param_grid = {
    "regressor__alpha": [0.001, 0.01, 0.1, 1, 10, 100]
}

In [25]:
ridge_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=ridge_param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv_reg,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)

In [26]:
ridge_search.fit(X_train_reg, y_train_reg)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...r', Ridge())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'regressor__alpha': [0.001, 0.01, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be dete

In [27]:
print("Best parameters:")
print(ridge_search.best_params_)

print("\nBest CV MAE:")
print(-ridge_search.best_score_)

Best parameters:
{'regressor__alpha': 0.1}

Best CV MAE:
2.035118085555478


In [28]:
ridge_results = pd.DataFrame(ridge_search.cv_results_)

ridge_results = ridge_results[
    [
        "param_regressor__alpha",
        "mean_test_score",
        "std_test_score",
        "mean_train_score"
    ]
].copy()

ridge_results["mean_test_score"] *= -1
ridge_results["mean_train_score"] *= -1

ridge_results = ridge_results.rename(
    columns={
        "mean_test_score": "Mean CV MAE",
        "std_test_score": "Std CV MAE",
        "mean_train_score": "Mean Train MAE"
    }
)

display(ridge_results.sort_values("Mean CV MAE"))

,param_regressor__alpha,Mean CV MAE,Std CV MAE,Mean Train MAE
2,0.100,2.035118,0.171627,1.985098
3,1.000,2.035127,0.171404,1.985489
1,0.010,2.035133,0.171648,1.985067
0,0.001,2.035134,0.171650,1.985064
4,10.000,2.043030,0.166698,1.994020
5,100.000,2.209300,0.125366,2.178237


### Ridge Regression Tuning Results

GridSearchCV selected **Ridge Regression with alpha = 0.1**, achieving a mean cross-validation MAE of approximately **2.0351 hours**.

The first four alpha values (0.001, 0.01, 0.1, and 1) produced nearly identical validation errors. This suggests that Ridge Regression's performance is relatively insensitive to regularization strength within this range.

Larger alpha values, particularly 100, increased the validation MAE. This suggests that excessive regularization can restrict the model's ability to capture relationships in the data.

We select alpha = 0.1 because it achieved the lowest measured CV MAE, while recognizing that its improvement over alpha = 1 is practically negligible.

In [29]:
from sklearn.metrics import mean_squared_error

best_reg = ridge_search.best_estimator_

y_pred_reg = best_reg.predict(X_test_reg)

test_mae = mean_absolute_error(y_test_reg, y_pred_reg)
test_rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))

print(f"Test MAE:  {test_mae:.4f} hours")
print(f"Test RMSE: {test_rmse:.4f} hours")

Test MAE:  2.2301 hours
Test RMSE: 3.4028 hours


### Final Ridge Regression Evaluation

After selecting the best hyperparameters using cross-validation, I evaluated the refitted Ridge Regression pipeline on the held-out test set.

The final results were:

- **Test MAE:** 2.2301 hours
- **Test RMSE:** 3.4028 hours

The test MAE indicates that predictions differ from actual resolution times by approximately 2.23 hours on average.

The test MAE is slightly higher than the best cross-validation MAE of 2.0351 hours. This difference is expected because the test set contains observations that were not used during model training or hyperparameter selection.

The RMSE is noticeably higher than the MAE. This suggests that the model makes some predictions with relatively large errors. Since RMSE squares prediction errors before averaging, large errors have a greater influence on this metric.

Overall, Ridge Regression provides a reasonable baseline for predicting resolution time, although its predictions may be less reliable for some individual tickets.

### Outliers and the Difference Between MAE and RMSE

The regression target has a maximum value of approximately 27.49 hours, compared with a median of approximately 5.39 hours. This indicates that some tickets take substantially longer to resolve than typical tickets.

MAE and RMSE respond differently to large prediction errors:

- MAE calculates the average absolute error and penalizes errors linearly.
- RMSE squares errors before averaging and therefore gives larger errors disproportionately more influence.

As a result, MAE and RMSE can favor different models. A model with fewer extreme prediction errors may achieve a better RMSE even if another model achieves a lower MAE.

For this experiment, MAE was the primary selection metric, as specified in the assignment. RMSE provides additional information about the impact of larger prediction errors.

## Final Assignment Reflections

**1. Why did we stratify the classification split?**

We used stratification to preserve approximately the same class distribution in both the training and test sets.

Our classification dataset is imbalanced, with approximately 78.8% non-escalated tickets and 21.2% escalated tickets.

Without stratification, random splitting could produce different class proportions, potentially leaving the training or test set with an unrepresentative number of minority-class examples.

Stratification helps ensure that both sets contain a representative distribution of the target classes, making training and evaluation more reliable.

**2. Why did we put preprocessing inside a Pipeline?**

We placed preprocessing inside a Pipeline to prevent data leakage and ensure consistent transformations during training and evaluation.

If preprocessing were performed before splitting the data, statistics such as medians and means could be calculated using information from the validation or test sets.

By including preprocessing inside the Pipeline, each cross-validation training fold independently learns its imputation values, scaling parameters, and categorical encoding. These learned transformations are then applied to the corresponding validation fold.

This ensures that the model is evaluated on data that did not influence the preprocessing or model fitting process.

**3. What does fold-to-fold variability tell us?**

The standard deviation of cross-validation scores measures how much model performance varies across different folds.

A low standard deviation indicates that a model performs relatively consistently across different training and validation splits. A high standard deviation indicates that performance is more sensitive to the particular data used in each fold, suggesting less stable generalization.

However, consistency alone does not guarantee accuracy. We must consider both the mean validation score and its standard deviation when comparing models.

**4. Why should we evaluate the test set only once?**

The test set should only be used for final evaluation because it provides an independent estimate of how well the selected model generalizes to unseen data.

If we repeatedly evaluate the test set and adjust model parameters based on its results, we indirectly use test-set information during model selection. This can cause the model to overfit to the test set, making its reported performance less trustworthy.

We therefore use cross-validation for model comparison and hyperparameter tuning, reserving the test set for one final evaluation after selecting the model.

**5. What deployment constraints might change our model choices?**

The choice of a model in a real-world system depends not only on its evaluation metrics but also on the consequences of its predictions.

For the escalation classifier, false positives create unnecessary escalations and consume staff resources, while false negatives allow tickets that should have been escalated to go unnoticed.

If missing an actual escalation is more costly, we may prioritize recall. If unnecessary escalations are more expensive, we may prioritize precision.

Other deployment considerations include:

- **Computational resources:** Whether the organization has enough resources to run the model efficiently.
- **Prediction speed:** Whether predictions must be generated in real time.
- **Interpretability:** Whether staff need to understand why a ticket was classified as an escalation.
- **Maintenance:** Whether the model can be monitored and retrained as ticket patterns change.
- **Data quality:** Whether the system can reliably handle missing values and previously unseen categories.

Therefore, the best model is not necessarily the one with the highest validation score. It is the one that provides an appropriate balance between predictive performance and real-world requirements.